## Epic 2 — Preprocessing Pipeline

Transforms the raw `MetObjects.csv` into a clean, model-ready feature matrix.
Covers null handling, column selection, and feature engineering for both the supervised (Department classification) and unsupervised (clustering) tracks.

In [1]:
import os

import numpy as np
import pandas as pd
import scipy.sparse
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

In [2]:
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PATH = os.path.join(ROOT, "data", "MetObjects.csv")

df_raw = pd.read_csv(DATA_PATH, low_memory=False)

print("Loaded:", df_raw.shape)

Loaded: (484956, 54)


In [3]:
# ── Column definitions (single source of truth) ────────────────────────────
# Source: reports/feature_engineering.md

COLS_TO_DROP = [
    # Identifiers
    "Object ID", "Object Number", "Constituent ID",
    # URLs — no feature signal
    "Link Resource", "Artist ULAN URL", "Artist Wikidata URL",
    "Object Wikidata URL", "Tags AAT URL", "Tags Wikidata URL",
    # 100% null / constant
    "Metadata Date", "Repository",
    # Geographic columns >90% null
    "River", "State", "Locus", "County", "Locale",
    "Excavation", "Subregion", "Region", "City",
    # Admin fields with no object-level signal
    "Rights and Reproduction", "Portfolio", "Gallery Number", "Credit Line",
    # Redundant / low-signal artist fields
    "Artist Prefix", "Artist Suffix", "Artist Alpha Sort",
    "Artist Display Bio", "Artist Display Name",
    # Flagged — dropping for now
    "Dynasty", "Reign", "Geography Type", "Country",
    "Artist Begin Date", "Artist End Date",
    "Dimensions", "Title", "Object Date",
]

COLS_NUMERIC = [
    "AccessionYear",          # 0.8% null → fill with median year
]

COLS_CATEGORICAL = [
    "Medium",                 # 1.5% null  — TF-IDF after comma-splitting
    "Tags",                   # 60.3% null — TF-IDF after pipe-splitting
    "Classification",         # 16.2% null — TF-IDF or top-N encode
    "Object Name",            # 0.5% null  — top-N frequency encode
    "Culture",                # 57.1% null — top-100 + "Other"
    "Period",                 # 81.2% null — top-N + "Other"
    "Artist Nationality",     # 41.7% null — top-N + "Other"
    "Artist Role",            # 41.7% null — label encode
    "Artist Gender",          # 78.0% null — binary flag
]

COLS_KEEP = [
    "Department",             # classification target — kept, encoded separately
    "Object Begin Date",      # int64, 0% null — clip + derive features
    "Object End Date",        # int64, 0% null — clip + derive features
    "Is Highlight",           # bool, 0% null
    "Is Timeline Work",       # bool, 0% null
    "Is Public Domain",       # bool, 0% null
]

print("Columns accounted for:",
      len(COLS_TO_DROP) + len(COLS_NUMERIC) + len(COLS_CATEGORICAL) + len(COLS_KEEP),
      "/ 54 total")

Columns accounted for: 54 / 54 total


In [6]:
df = df_raw.drop(columns=COLS_TO_DROP)

print(f"Shape before drop : {df_raw.shape}")
print(f"Shape after drop  : {df.shape}")
print(f"Columns removed   : {df_raw.shape[1] - df.shape[1]}")
print()
assert "Department" in df.columns, "ERROR: Department was accidentally dropped!"
print("Test - Department column intact!!")

Shape before drop : (484956, 54)
Shape after drop  : (484956, 16)
Columns removed   : 38

Test - Department column intact!!
